In [4]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
import tkinter as tk
from tkinter import ttk, filedialog
import pandas as pd
from datetime import datetime

In [6]:
class CSVReaderApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Lector de CSV")
        
        # Frame principal
        self.main_frame = ttk.Frame(root, padding="10")
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Área para arrastrar y soltar
        self.drop_area = tk.Label(
            self.main_frame, 
            text="Arrastra tu archivo CSV aquí",
            relief=tk.RAISED,
            width=50,
            height=10
        )
        self.drop_area.pack(pady=10)
        self.drop_area.bind("<Button-1>", self.browse_file)
        self.drop_area.bind("<B1-Motion>", self.drag_file)
        self.drop_area.bind("<ButtonRelease-1>", self.drop_file)
        
        # Botón para cargar archivo
        self.load_button = ttk.Button(
            self.main_frame,
            text="Cargar CSV",
            command=self.browse_file
        )
        self.load_button.pack(pady=5)
        
        # Tabla para mostrar datos
        self.tree = ttk.Treeview(self.main_frame)
        self.tree.pack(fill=tk.BOTH, expand=True)
        
    def browse_file(self, event=None):
        file_path = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
        if file_path:
            self.process_csv(file_path)
    
    def drag_file(self, event):
        # Puedes añadir efectos visuales durante el arrastre
        pass
    
    def drop_file(self, event):
        # Aquí implementarías la lógica para manejar el archivo soltado
        # Por ahora usaremos el mismo método que para el botón
        self.browse_file()
    
    def process_csv(self, file_path):
        try:
            # Leer CSV
            df = pd.read_csv(file_path)
            
            # Verificar columnas
            required_columns = ['nombre', 'apellidos', 'fecha de nacimiento', 'ciudad']
            if not all(col in df.columns for col in required_columns):
                raise ValueError("El CSV no tiene las columnas requeridas")
                
            # Procesar fechas
            df['fecha de nacimiento'] = pd.to_datetime(
                df['fecha de nacimiento'], 
                format='%m/%d/%Y'
            )
            
            # Agregar ID secuencial si no existe
            if 'id' not in df.columns:
                df.insert(0, 'id', range(1, len(df)+1))
            
            # Mostrar datos en la tabla
            self.display_data(df)
            
        except Exception as e:
            tk.messagebox.showerror("Error", f"Error al procesar el CSV: {str(e)}")
    
    def display_data(self, df):
        # Limpiar tabla
        for i in self.tree.get_children():
            self.tree.delete(i)
        
        # Configurar columnas
        self.tree["columns"] = list(df.columns)
        for col in df.columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=100)
        
        # Insertar datos
        for _, row in df.iterrows():
            self.tree.insert("", tk.END, values=list(row))

if __name__ == "__main__":
    root = tk.Tk()
    app = CSVReaderApp(root)
    root.mainloop()